# Static trajectory encoding projection matrix

Construct the shared per-axis matrix $A \in \mathbb{R}^{96 \times 25}$ that decodes a static NF trajectory encoding into its 96 target displacement coordinates.

The matrix is generated by passing standard basis vectors through the same task organization and Matérn interpolation calculations used by `AntFiniteMaternWrapper`. For a batched 50-dimensional encoding, reshape it to `(..., 2, 25)` and apply the same matrix to both x and y.

In [1]:
from pathlib import Path
import sys

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
from brax import envs

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jax_rq_nsf").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from jax_rq_nsf import FlowConfig, NormalizingFlow
from task_wrappers.nf_trajectory_following import AntFiniteMaternWrapper

WAY_POINTS = 12
STEPS_PER_WAY_POINT = 8
AXIS_ENCODING_DIMENSION = 2 * WAY_POINTS + 1
ROLLOUT_LENGTH = WAY_POINTS * STEPS_PER_WAY_POINT
OUTPUT_PATH = PROJECT_ROOT / "output/mapping_test/trajectory_projection_matrix.npy"

flow = NormalizingFlow(
    FlowConfig(
        dimension=2 * AXIS_ENCODING_DIMENSION,
        num_layers=8,
        hidden_features=(128, 128),
        num_bins=16,
        tail_bound=6.0,
    )
)
base_env = envs.create(
    env_name="ant",
    episode_length=4096,
    backend="mjx",
    reset_noise_scale=0.0,
)
env = AntFiniteMaternWrapper(
    base_env,
    flow,
    way_points=WAY_POINTS,
    steps_per_way_point=STEPS_PER_WAY_POINT,
    inner_radius=0.5,
    max_radius=2.0,
    tolerance_radius=0.1,
)

print("Per-axis encoding dimension:", AXIS_ENCODING_DIMENSION)
print("Rollout length:", ROLLOUT_LENGTH)

Per-axis encoding dimension: 25
Rollout length: 96


/home_nfs/runjun/python_envs/jax/lib/python3.10/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [2]:
def interpolation_step(task_state, _):
    cycle_t = task_state.cycle_t + env.dt
    base_values = jnp.asarray(env._base_fn(cycle_t), dtype=jnp.float64)
    interpolation_matrix = jnp.asarray(
        env.Sigma22_inv_S,
        dtype=jnp.float64,
    )
    shifted_positions = jnp.sum(
        (task_state.reshaped_sequence @ interpolation_matrix) * base_values,
        axis=-1,
    )
    return task_state.replace(cycle_t=cycle_t), shifted_positions[:, 0]


def interpolation_period(task_state, _):
    task_state, positions = jax.lax.scan(
        interpolation_step,
        task_state,
        xs=None,
        length=STEPS_PER_WAY_POINT,
    )
    shifted_sequence = jnp.concatenate(
        [task_state.reshaped_sequence[:, 1:, :], task_state.padding_element],
        axis=1,
    )
    task_state = task_state.replace(
        cycle_t=task_state.cycle_t - env.period_t,
        reshaped_sequence=shifted_sequence,
    )
    return task_state, positions


def decode_static_encoding(z):
    """Decode one 50-D static encoding to a (96, 2) displacement path."""
    sampled_sequence = jnp.asarray(z, dtype=jnp.float64).reshape(
        2,
        AXIS_ENCODING_DIMENSION,
    )
    task_state = env._organize_z_state(
        sampled_sequence,
        position_offset=jnp.zeros((2,), dtype=jnp.float64),
    )
    _, position_blocks = jax.lax.scan(
        interpolation_period,
        task_state,
        xs=None,
        length=WAY_POINTS,
    )
    return position_blocks.reshape(ROLLOUT_LENGTH, 2)


decode_static_encoding = jax.jit(decode_static_encoding)

In [6]:
axis_basis = jnp.eye(AXIS_ENCODING_DIMENSION, dtype=jnp.float64)


def decode_x_basis(axis_encoding):
    static_encoding = jnp.zeros(
        (2, AXIS_ENCODING_DIMENSION),
        dtype=axis_encoding.dtype,
    )
    static_encoding = static_encoding.at[0].set(axis_encoding)
    return decode_static_encoding(static_encoding.reshape(-1))[:, 0]


# vmap produces (basis_input, rollout_step), so transpose it into the
# conventional linear-map layout (rollout_step, axis_encoding).
A = jax.vmap(decode_x_basis)(axis_basis).T
assert A.shape == (ROLLOUT_LENGTH, AXIS_ENCODING_DIMENSION)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_PATH, np.asarray(A, dtype=np.float32), allow_pickle=False)

print("A shape:", A.shape)
print("A dtype:", A.dtype)
print("Saved:", OUTPUT_PATH)

A shape: (96, 25)
A dtype: float64
Saved: /home_nfs/runjun/projects/TBQDRL/output/mapping_test/trajectory_projection_matrix.npy


In [7]:
key = jax.random.PRNGKey(8848)
test_z = jax.random.normal(
    key,
    (32, 2 * AXIS_ENCODING_DIMENSION),
    dtype=jnp.float64,
)
direct_paths = jax.vmap(decode_static_encoding)(test_z)

reshaped_z = test_z.reshape(-1, 2, AXIS_ENCODING_DIMENSION)
matrix_paths = jnp.einsum("td,bcd->btc", A, reshaped_z)

maximum_error = jnp.max(jnp.abs(direct_paths - matrix_paths))
np.testing.assert_allclose(
    np.asarray(matrix_paths),
    np.asarray(direct_paths),
    rtol=1e-10,
    atol=1e-10,
)
print("Maximum direct-vs-matrix decoding error:", float(maximum_error))

Maximum direct-vs-matrix decoding error: 1.1457501614131615e-13


In [8]:
# Mapping-training usage. mapped_z has shape (..., 50), while y and scale
# have shapes (..., 96, 2) and (..., 96, 1), respectively.
def decode_with_A(mapped_z, projection_matrix=A):
    per_axis_z = mapped_z.reshape(
        mapped_z.shape[:-1] + (2, AXIS_ENCODING_DIMENSION)
    )
    return jnp.einsum("td,...cd->...tc", projection_matrix, per_axis_z)


def trajectory_discrepancy(mapped_z, y, scale, projection_matrix=A):
    target_y = decode_with_A(mapped_z, projection_matrix)
    return jnp.mean(
        jnp.square(y - target_y) * scale + (1.0 - scale) * 25.0,
        axis=(-1, -2),
    )